# Segmentación de Vasos Retinianos — `colab_runner.ipynb`

Notebook para ejecutar el proyecto de la **Pregunta 2 - Segmentación de Vasos Retinianos para Detección de Retinopatía Diabética(U-Net)** en Google Colab usando los datasets montados desde Google Drive.

Antes de ejecutar:

1. En Colab selecciona **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU**.
2. Ejecuta las celdas en orden.
3. Verifica que tus datasets estén en:

```text
/content/drive/MyDrive/EP01/pregunta2/datasets/DRIVE
/content/drive/MyDrive/EP01/pregunta2/datasets/CHASE_DB1/new/chase/test/test
/content/drive/MyDrive/EP01/pregunta2/datasets/STARE
```


## 1. Montar Google Drive, clonar repositorio y crear enlaces a datasets

Esta celda:

- monta Google Drive;
- clona o actualiza el repositorio;
- detecta automáticamente si DRIVE está en `DRIVE/training/...` o `DRIVE/datasets/training/...`;
- crea los enlaces simbólicos dentro de `data/`;
- muestra conteos de archivos para verificar que las rutas son correctas.


In [ ]:
import os
import shutil
from pathlib import Path

# ─────────────────────────────────────────────────────────────
# Configuración principal
# ─────────────────────────────────────────────────────────────
REPO_URL = "https://github.com/manadevelop/retinal_segmentation.git"
PROJECT_DIR = Path("/content/retinal_segmentation")
DATASETS_BASE = Path("/content/drive/MyDrive/EP01/pregunta2/datasets")

# ─────────────────────────────────────────────────────────────
# Montar Google Drive
# ─────────────────────────────────────────────────────────────
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("No se pudo montar Google Drive automáticamente:", e)

# ─────────────────────────────────────────────────────────────
# Clonar o actualizar repositorio
# ─────────────────────────────────────────────────────────────
if not PROJECT_DIR.exists():
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    !cd {PROJECT_DIR} && git pull

os.chdir(PROJECT_DIR)
print("Directorio actual:", Path.cwd())

# ─────────────────────────────────────────────────────────────
# Verificar GPU
# ─────────────────────────────────────────────────────────────
print("\nVerificando GPU disponible:")
!nvidia-smi

# ─────────────────────────────────────────────────────────────
# Rutas reales en Google Drive
# ─────────────────────────────────────────────────────────────
DRIVE_BASE = DATASETS_BASE / "DRIVE"
CHASE_BASE = DATASETS_BASE / "CHASE_DB1/new/chase/test/test"
STARE_BASE = DATASETS_BASE / "STARE"

# DRIVE puede venir como:
#   DRIVE/training/images
# o como:
#   DRIVE/datasets/training/images
if (DRIVE_BASE / "training" / "images").exists():
    DRIVE_SRC = DRIVE_BASE
elif (DRIVE_BASE / "datasets" / "training" / "images").exists():
    DRIVE_SRC = DRIVE_BASE / "datasets"
else:
    raise FileNotFoundError(
        "No se encontró la estructura esperada de DRIVE. Debe existir una de estas rutas:\n"
        f"  {DRIVE_BASE / 'training' / 'images'}\n"
        f"  {DRIVE_BASE / 'datasets' / 'training' / 'images'}"
    )

print("\nRaíces detectadas:")
print("DRIVE_SRC :", DRIVE_SRC)
print("CHASE_BASE:", CHASE_BASE)
print("STARE_BASE:", STARE_BASE)

# ─────────────────────────────────────────────────────────────
# Función segura para crear symlinks
# ─────────────────────────────────────────────────────────────
def make_link(src, dst, required=True):
    src = Path(src)
    dst = Path(dst)

    dst.parent.mkdir(parents=True, exist_ok=True)

    if dst.is_symlink() or dst.is_file():
        dst.unlink()
    elif dst.exists():
        shutil.rmtree(dst)

    if not src.exists():
        msg = f"No encontrado en Google Drive: {src}"
        if required:
            raise FileNotFoundError(msg)
        print("⚠", msg)
        return False

    os.symlink(str(src), str(dst))
    print(f"✓ {dst} -> {src}")
    return True

# ─────────────────────────────────────────────────────────────
# Crear estructura data/
# ─────────────────────────────────────────────────────────────
Path("data").mkdir(exist_ok=True)

print("\nCreando enlaces simbólicos:")

# DRIVE
make_link(DRIVE_SRC, "data/drive", required=True)

# CHASE_DB1
make_link(CHASE_BASE / "images", "data/chase_db1/images", required=True)
make_link(CHASE_BASE / "1st_manual", "data/chase_db1/labels", required=True)
make_link(CHASE_BASE / "mask", "data/chase_db1/mask", required=False)
make_link(CHASE_BASE / "2nd_manual", "data/chase_db1/labels-2nd", required=False)

# STARE
make_link(STARE_BASE / "images", "data/stare/images", required=False)
make_link(STARE_BASE / "masks", "data/stare/masks", required=False)
make_link(STARE_BASE / "masks", "data/stare/labels-ah", required=False)

print("\nEstructura montada:")
for p in sorted(Path("data").glob("**/*")):
    if p.is_symlink() or p.is_dir():
        print(" ", p, "->", os.readlink(p) if p.is_symlink() else "")

# ─────────────────────────────────────────────────────────────
# Conteo de archivos
# ─────────────────────────────────────────────────────────────
def count_files(path):
    path = Path(path)
    if not path.exists():
        return 0
    return sum(1 for p in path.glob("*") if p.is_file())

checks = {
    "DRIVE train images": Path("data/drive/training/images"),
    "DRIVE train manual": Path("data/drive/training/1st_manual"),
    "DRIVE train FOV": Path("data/drive/training/mask"),
    "DRIVE test images": Path("data/drive/test/images"),
    "DRIVE test manual": Path("data/drive/test/1st_manual"),
    "DRIVE test FOV": Path("data/drive/test/mask"),
    "CHASE images": Path("data/chase_db1/images"),
    "CHASE manual": Path("data/chase_db1/labels"),
    "CHASE FOV": Path("data/chase_db1/mask"),
    "CHASE 2nd manual": Path("data/chase_db1/labels-2nd"),
    "STARE images": Path("data/stare/images"),
    "STARE masks": Path("data/stare/masks"),
}

print("\nConteo de archivos detectados:")
for name, path in checks.items():
    print(f"{name:<22} existe={str(path.exists()):<5} archivos={count_files(path):<3} ruta={path}")

# Advertencia específica para DRIVE test
if count_files("data/drive/test/1st_manual") == 0:
    print("\n⚠ ADVERTENCIA IMPORTANTE:")
    print("No se encontró data/drive/test/1st_manual con máscaras de vasos.")
    print("La carpeta data/drive/test/mask contiene FOV, no ground truth de vasos.")
    print("Si el pipeline evalúa DRIVE test con métricas, necesitarás agregar test/1st_manual o ajustar la evaluación.")


## 2. Validar archivos del proyecto e imports Python

Esta celda corrige el problema típico de Colab donde `PYTHONPATH` no actualiza el `sys.path` del kernel actual. También evita que la carpeta `data/` de datasets oculte al paquete Python `src/data/`.


In [ ]:
import os
import sys
from pathlib import Path

PROJECT_DIR = Path("/content/retinal_segmentation")
SRC_DIR = PROJECT_DIR / "src"

os.chdir(PROJECT_DIR)

# Variables para procesos nuevos
os.environ["DATA_FROM_DRIVE"] = "1"
os.environ["PYTHONPATH"] = f"{SRC_DIR}:{os.environ.get('PYTHONPATH', '')}"

# Corregir sys.path para el kernel actual de Colab
src_str = str(SRC_DIR)
if src_str in sys.path:
    sys.path.remove(src_str)
sys.path.insert(0, src_str)

# Limpiar imports previos incorrectos de data.*
for module_name in list(sys.modules.keys()):
    if module_name == "data" or module_name.startswith("data."):
        del sys.modules[module_name]

print("Directorio actual:", Path.cwd())
print("DATA_FROM_DRIVE:", os.environ["DATA_FROM_DRIVE"])
print("PYTHONPATH:", os.environ["PYTHONPATH"])

print("\nVerificando archivos principales:")
required_files = [
    PROJECT_DIR / "requirements.txt",
    PROJECT_DIR / "run_all.sh",
    SRC_DIR / "train.py",
    SRC_DIR / "data" / "__init__.py",
    SRC_DIR / "data" / "dataset.py",
    SRC_DIR / "data" / "transforms.py",
]

for f in required_files:
    print("✓" if f.exists() else "✗", f)
    if not f.exists():
        raise FileNotFoundError(f"Falta archivo requerido: {f}")

print("\nPrimeras rutas de sys.path:")
for p in sys.path[:8]:
    print(" ", p)

print("\nValidando sintaxis de run_all.sh:")
!bash -n run_all.sh && echo "✓ Sintaxis de run_all.sh OK"

print("\nImportando clases del proyecto:")
from data.dataset import DriveDataset, ChaseDB1Dataset, StareDataset
from data.transforms import ValTransform

print("✓ Imports correctos desde src/data")


## 3. Validación opcional de datasets

Esta celda intenta cargar una muestra de DRIVE, CHASE_DB1 y STARE. Si falla, el mensaje indica exactamente qué dataset o ruta está fallando.


In [ ]:
from pathlib import Path

# Asegurar imports después de la celda anterior
from data.dataset import DriveDataset, ChaseDB1Dataset, StareDataset
from data.transforms import ValTransform

tr = ValTransform(img_size=512)

print("Validando DRIVE train...")
drive_train = DriveDataset(
    root="data/drive",
    split="train",
    transform=tr,
)
x, y, fov = drive_train[0]
print(
    f"✓ DRIVE train: n={len(drive_train)}, "
    f"image={tuple(x.shape)}, mask_sum={float(y.sum()):.1f}, fov_sum={float(fov.sum()):.1f}"
)

print("\nValidando CHASE_DB1 all...")
chase_all = ChaseDB1Dataset(
    root="data/chase_db1",
    split="all",
    transform=tr,
)
x, y, fov = chase_all[0]
print(
    f"✓ CHASE all: n={len(chase_all)}, "
    f"image={tuple(x.shape)}, mask_sum={float(y.sum()):.1f}, fov_sum={float(fov.sum()):.1f}"
)

print("\nValidando STARE ah...")
try:
    stare = StareDataset(
        root="data/stare",
        annotator="ah",
        transform=tr,
    )
    x, y, fov = stare[0]
    print(
        f"✓ STARE ah: n={len(stare)}, "
        f"image={tuple(x.shape)}, mask_sum={float(y.sum()):.1f}, fov_sum={float(fov.sum()):.1f}"
    )
except Exception as e:
    print("⚠ STARE no se validó:", e)

print("\nValidación terminada.")


## 4. Ejecutar pipeline completo

Esta celda ejecuta el proyecto. Se fuerza `DATA_FROM_DRIVE=1` y `PYTHONPATH=/content/retinal_segmentation/src` directamente en el comando para evitar errores de entorno.


In [ ]:
import os
from pathlib import Path

PROJECT_DIR = Path("/content/retinal_segmentation")
SRC_DIR = PROJECT_DIR / "src"
os.chdir(PROJECT_DIR)

print("Ejecutando pipeline completo...")
print("DATA_FROM_DRIVE=1")
print("PYTHONPATH=", SRC_DIR)

!DATA_FROM_DRIVE=1 PYTHONPATH="{SRC_DIR}:$PYTHONPATH" bash run_all.sh


## 5. Respaldo opcional en Google Drive

Ejecuta esta celda al finalizar para copiar `outputs/`, `results/` y `reports/` a Google Drive.


In [ ]:
import os
import shutil
from pathlib import Path

PROJECT_DIR = Path("/content/retinal_segmentation")
os.chdir(PROJECT_DIR)

BACKUP_DIR = Path("/content/drive/MyDrive/retinal_segmentation_backup")
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

for folder in ["outputs", "results", "reports"]:
    src = PROJECT_DIR / folder
    dst = BACKUP_DIR / folder

    if src.exists():
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f"✓ Respaldado: {src} -> {dst}")
    else:
        print(f"⚠ No existe {src}; se omite.")

print("Respaldo finalizado.")


## 6. Resumen opcional de métricas

Busca archivos `test_metrics.json` dentro de `outputs/` y muestra una tabla simple.


In [ ]:
import json
from pathlib import Path

outputs_dir = Path("/content/retinal_segmentation/outputs")

if not outputs_dir.exists():
    print("No existe outputs/. Ejecuta primero el pipeline.")
else:
    metric_files = sorted(outputs_dir.glob("**/test_metrics.json"))

    if not metric_files:
        print("No se encontraron archivos test_metrics.json dentro de outputs/.")
    else:
        print(f"{'Experimento':<45} {'Sens':>8} {'Spec':>8} {'F1':>8} {'AUC':>8}")
        print("-" * 85)

        for mf in metric_files:
            with open(mf, "r") as f:
                m = json.load(f)

            exp = str(mf.parent.relative_to(outputs_dir))
            print(
                f"{exp:<45} "
                f"{m.get('sensibilidad', m.get('sensitivity', 0)):>8.4f} "
                f"{m.get('especificidad', m.get('specificity', 0)):>8.4f} "
                f"{m.get('f1', m.get('dice', 0)):>8.4f} "
                f"{m.get('auc_roc', m.get('auc', 0)):>8.4f}"
            )
